# 📦 W10: 공급업체 소통 자동화
**hexa-2 — W10** | ⏱️ 60분 | 🎯 이슈유형(품질불량/납품지연/발주확인)→공문 Gemini생성→ZIP

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q google-generativeai pandas matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, pandas as pd
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
model=genai.GenerativeModel('gemini-2.0-flash'); print('✅ 준비 완료')


## Step 1: 가게 정보 입력 (✏️)

In [ ]:
INFO={'name':'✏️ [가게명]','type':'✏️ [한식|중식|일식|분식]'}
print('✅',INFO['name'])


## Step 2: 데이터 준비

In [ ]:
import io, requests
URL='https://raw.githubusercontent.com/Reasonofmoon/hexa-2/main/shared/fnb_sales_sample.csv'
try:
    df=pd.read_csv(URL,encoding='utf-8-sig'); print(f'✅ GitHub 로드: {len(df)}행')
except:
    try:
        from google.colab import files; up=files.upload()
        if up: df=pd.read_csv(io.BytesIO(list(up.values())[0]),encoding='utf-8-sig')
        else: raise Exception('취소됨')
    except:
        df=pd.DataFrame({'날짜':['2026-01-01']*3,'메뉴명':['김치찌개','된장찌개','순두부'],'주문수':[150,100,80],'매출':[750000,500000,400000],'평점':[4.5,4.2,4.7]})
        print('📋 기본 샘플 데이터 사용')
print(df.head())


## Step 3: 이슈유형(품질불량/납품지연/발주확인)→공문 Gemini생성→ZIP

In [ ]:
import numpy as np
np.random.seed(42)
p=f"""이슈유형(품질불량/납품지연/발주확인)→공문 Gemini생성→ZIP F&B 컨설턴트로서 수행하세요.
가게: {{INFO['name']}} ({{INFO['type']}})
실용적 인사이트 3가지 + 즉시 실행 액션. 마크다운."""
resp=model.generate_content(p)
print(resp.text)


## Step 4: 시각화

In [ ]:
if '날짜' in df.columns and '매출' in df.columns:
    import matplotlib.pyplot as plt
    fig,ax=plt.subplots(figsize=(10,4))
    menu_sales=df.groupby('메뉴명')['매출'].sum() if '메뉴명' in df.columns else df['매출']
    menu_sales.plot(kind='bar',ax=ax,color='steelblue')
    ax.set_title(f'{INFO["name"]} — {title}')
    ax.set_ylabel('매출(원)'); plt.xticks(rotation=30)
    plt.tight_layout(); plt.savefig('chart.png',dpi=150,bbox_inches='tight'); plt.show()
else: print('차트 생략 (데이터 컬럼 불일치)')


## Step 5: 다운로드

In [ ]:
with open('report.md','w',encoding='utf-8') as f: f.write(resp.text)
from google.colab import files
files.download('report.md')
try: files.download('chart.png')
except: pass
print('✅ 완료!')


---
## 🔥 확장 과제
다음 주차와 연계하여 통합 분석을 시도하세요.